In [45]:
import scrape_nbadraft_net
from sklearn.feature_selection import mutual_info_regression

import pandas as pd

In [4]:
df = scrape_nbadraft_net.build_df()

In [8]:
df.isna().sum()


nbadraft_net_comps     85
descr_raw               0
Athleticism            36
Size                   36
Defense                36
Strength               36
Quickness              36
Leadership             36
Jump Shot              37
NBA Ready              36
Rebounding            711
Potential              36
Post Skills           711
Intangibles            36
mock                  154
big_board              43
overall                36
Name                    0
Ball Handling         368
Passing               369
ows                   243
dws                   243
vorp                  243
per                   243
g                     243
dtype: int64

In [9]:
played_in_league = df[~df.g.isna()]

In [11]:
played_in_league.isna().sum()


nbadraft_net_comps     62
descr_raw               0
Athleticism            17
Size                   17
Defense                17
Strength               17
Quickness              17
Leadership             17
Jump Shot              17
NBA Ready              17
Rebounding            545
Potential              17
Post Skills           545
Intangibles            17
mock                   85
big_board              20
overall                17
Name                    0
Ball Handling         261
Passing               264
ows                     0
dws                     0
vorp                    0
per                     0
g                       0
dtype: int64

In [12]:
df_backup = df.copy()

Let's look at the mutual information...

In [13]:
with_ratings = played_in_league[~played_in_league.Athleticism.isna()]

In [14]:
with_ratings.isna().sum()


nbadraft_net_comps     47
descr_raw               0
Athleticism             0
Size                    0
Defense                 0
Strength                0
Quickness               0
Leadership              0
Jump Shot               0
NBA Ready               0
Rebounding            528
Potential               0
Post Skills           528
Intangibles             0
mock                   74
big_board              17
overall                 0
Name                    0
Ball Handling         244
Passing               247
ows                     0
dws                     0
vorp                    0
per                     0
g                       0
dtype: int64

In [114]:
predictors = ['Athleticism', 'Size', 'Defense', 'Strength', 'Quickness', 'Leadership', 'Jump Shot', 
'NBA Ready', 'Potential', 'Intangibles', 'overall']

X = with_ratings[predictors].copy()
# these columns should all be able to be converted to ints
X[X.columns] = X[X.columns].apply(lambda x: pd.to_numeric(x, errors='coerce', downcast='integer'))

VORP and other metrics are the median of the season-level values. I didn't want to do totals because injuries and small sample size.

'g' -- number of games played in, is a sum.

In [115]:
X

,Athleticism,Size,Defense,Strength,Quickness,Leadership,Jump Shot,NBA Ready,Potential,Intangibles,overall
0,9,8,8,7,8,8,6,8,8,9,92
1,7,6,8,8,8,8,8,8,7,8,91
2,7,9,8,8,7,8,9,8,7,8,92
4,8,8,7,7,8,7,8,8,7,7,89
6,9,9,8,7,9,8,8,7,9,7,96
...,...,...,...,...,...,...,...,...,...,...,...
1009,7,8,7,7,7,8,8,7,7,8,90
1010,10,8,8,7,8,7,7,7,9,7,92
1011,7,10,8,5,8,6,7,6,9,6,86
1012,8,9,8,7,8,7,8,7,9,8,95


In [20]:
y = with_ratings.vorp

In [32]:
y


0       0.95
1       0.10
2      -0.25
4       0.30
6      -1.00
        ... 
1009   -0.10
1010   -0.10
1011   -0.15
1012   -0.30
1013    1.90
Name: vorp, Length: 754, dtype: float64

In [35]:
y


0       0.95
1       0.10
2      -0.25
4       0.30
6      -1.00
        ... 
1009   -0.10
1010   -0.10
1011   -0.15
1012   -0.30
1013    1.90
Name: vorp, Length: 754, dtype: float64

## Mutual Information

cribbed from https://www.kaggle.com/code/ryanholbrook/mutual-information

when comparing to VORP, the results vary widely between runs

when comparing to 'g', the total number of games played, it's much more stable

however 'g' isn't a great measure, because players drafted eg. last year will have fewer games through no fault of their own.

In [94]:
mi_scores = mutual_info_regression(X,with_ratings['ows'], discrete_features=(X.dtypes == 'int'))
mi_scores = pd.Series(mi_scores, name="MI Scores", index=X.columns)
mi_scores.sort_values(ascending=False)

Leadership     0.055143
Intangibles    0.051476
overall        0.049286
NBA Ready      0.028711
Jump Shot      0.023881
Athleticism    0.000000
Size           0.000000
Defense        0.000000
Quickness      0.000000
Strength       0.000000
Potential      0.000000
Name: MI Scores, dtype: float64

In [116]:
with_ratings['ws'] = with_ratings['ows'] + with_ratings['dws']

C:\Users\casey\AppData\Local\Temp\ipykernel_24872\1051492653.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  with_ratings['ws'] = with_ratings['ows'] + with_ratings['dws']


In [118]:
with_ratings.g.corr(with_ratings['big_board'])

np.float64(-0.35106105417358374)

In [89]:
with_ratings.columns

Index(['nbadraft_net_comps', 'descr_raw', 'Athleticism', 'Size', 'Defense',
       'Strength', 'Quickness', 'Leadership', 'Jump Shot', 'NBA Ready',
       'Rebounding', 'Potential', 'Post Skills', 'Intangibles', 'mock',
       'big_board', 'overall', 'Name', 'Ball Handling', 'Passing', 'ows',
       'dws', 'vorp', 'per', 'g'],
      dtype='object')